# Phase 0 — Acceptor-Recognition Circuit Pilot (GPU / Colab)

Standalone GPU copy of `experiments/phase0_pilot.py` from the
[acceptor-interpretability](.) project — no repo clone needed, everything
is inlined so this runs on a fresh Colab GPU runtime.

Measures `m_acc` (masked-AG logit margin, SpliceBERT) and `Δppt` on real
3' splice-site windows from `leannmlindsey/GUE` (`splice_reconstructed`,
GENCODE/DeepSplice-derived), across: clean, branch-point (BP) disrupted,
polypyrimidine-tract (PPT) disrupted, both disrupted, a bare random-AG
control, and a downstream decoy AG — plus a matched non-splice control set.

**Runtime > Change runtime type > GPU** before running. Everything auto-
detects `cuda` and falls back to CPU if none is available.

In [ ]:
!pip install -q multimolecule


In [ ]:
import random as pyrandom
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from multimolecule import RnaTokenizer, SpliceBertForMaskedLM

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

# --- config (mirrors configs/{model,experiment}.yaml) -----------------------
CHECKPOINT = "multimolecule/splicebert"

DATASET, CONFIG, SPLIT = "leannmlindsey/GUE", "splice_reconstructed", "test"
CSV_URL = f"https://huggingface.co/datasets/{DATASET}/resolve/main/GUE/{CONFIG}/{SPLIT}.csv"
SEQ_LEN = 400
ACCEPTOR_LABEL, DONOR_LABEL, NEGATIVE_LABEL = 0, 1, 2

AG_START, AG_LEN = 198, 2
PPT_START, PPT_END = 183, 198
BP_START, BP_END = 160, 183
DECOY_FROM, DECOY_TO = 201, 280

N_ACCEPTOR = 500   # GPU can afford more samples than the CPU default (200)
N_NEGATIVE = 500
SEED = 0
N_BOOT = 5000
ALPHA = 0.05

BASES = "ACGU"


In [ ]:
# --- data: load GUE split, split into acceptor / negative subsets -----------
print(f"loading GUE/{CONFIG} ({SPLIT}) ...")
_df = pd.read_csv(CSV_URL)
_df = _df[_df["sequence"].str.len() == SEQ_LEN].reset_index(drop=True)

_acc_mask = _df["label"] == ACCEPTOR_LABEL
_has_ag = _df["sequence"].str[AG_START:AG_START + AG_LEN] == "AG"
acceptor_df = _df[_acc_mask & _has_ag].reset_index(drop=True)
negative_df = _df[_df["label"] == NEGATIVE_LABEL].reset_index(drop=True)

acceptor_df = acceptor_df.sample(n=min(N_ACCEPTOR, len(acceptor_df)), random_state=SEED).reset_index(drop=True)
negative_df = negative_df.sample(n=min(N_NEGATIVE, len(negative_df)), random_state=SEED).reset_index(drop=True)
print(f"{len(acceptor_df)} acceptor windows, {len(negative_df)} non-splice control windows")


In [ ]:
# --- model: SpliceBERT masked-LM, single-nucleotide tokenizer ---------------
# raw sequence index i -> token index i+1 (leading <cls>)
print(f"loading {CHECKPOINT} ...")
tokenizer = RnaTokenizer.from_pretrained(CHECKPOINT)
model = SpliceBertForMaskedLM.from_pretrained(CHECKPOINT).to(DEVICE).eval()
MASK_ID = tokenizer.mask_token_id
BASE_IDS = {b: tokenizer.convert_tokens_to_ids(b) for b in BASES}


@torch.no_grad()
def masked_logits_at(sequence: str, raw_positions):
    enc = tokenizer(sequence, return_tensors="pt").to(DEVICE)
    input_ids = enc["input_ids"].clone()
    token_positions = [p + 1 for p in raw_positions]
    for tp in token_positions:
        input_ids[0, tp] = MASK_ID
    enc["input_ids"] = input_ids
    out = model(**enc)
    return out.logits[0, token_positions]


def _margin(logits_row, true_base: str) -> float:
    true_logit = logits_row[BASE_IDS[true_base]].item()
    other = [BASE_IDS[b] for b in BASES if b != true_base]
    return true_logit - logits_row[other].max().item()


def m_acc(sequence: str, ag_start: int) -> float:
    logits = masked_logits_at(sequence, [ag_start, ag_start + 1])
    return _margin(logits[0], "A") + _margin(logits[1], "G")


In [ ]:
# --- corruption / control construction (content-destroying, no injected AG) --
def _no_ag_choices(prev_base):
    return "ACU" if prev_base == "A" else BASES


def random_no_ag(length: int, rng: pyrandom.Random, prev_base=None) -> str:
    out, prev = [], prev_base
    for _ in range(length):
        c = rng.choice(_no_ag_choices(prev))
        out.append(c)
        prev = c
    return "".join(out)


def _fix_right_boundary(region: str, next_base, rng: pyrandom.Random) -> str:
    if next_base != "G" or region[-1] != "A":
        return region
    prev = region[-2] if len(region) > 1 else None
    choices = [b for b in _no_ag_choices(prev) if b != "A"]
    return region[:-1] + rng.choice(choices)


def corrupt_region(sequence: str, start: int, end: int, rng: pyrandom.Random) -> str:
    prev_base = sequence[start - 1] if start > 0 else None
    next_base = sequence[end] if end < len(sequence) else None
    region = random_no_ag(end - start, rng, prev_base=prev_base)
    region = _fix_right_boundary(region, next_base, rng)
    return sequence[:start] + region + sequence[end:]


def random_ag_variant(sequence: str, ag_start: int, ag_len: int, rng: pyrandom.Random) -> str:
    body = random_no_ag(len(sequence), rng)
    return body[:ag_start] + "AG" + body[ag_start + ag_len:]


def find_decoy_ag(sequence: str, search_from: int, search_to: int):
    end = min(search_to, len(sequence) - 1)
    for i in range(search_from, end):
        if sequence[i:i + 2] == "AG":
            return i
    return None


In [ ]:
# --- run the pilot ------------------------------------------------------
def score_acceptor_sample(seq: str) -> dict:
    rng = pyrandom.Random(hash(seq) & 0xFFFFFFFF)
    clean = m_acc(seq, AG_START)
    bp_only = m_acc(corrupt_region(seq, BP_START, BP_END, rng), AG_START)
    ppt_only = m_acc(corrupt_region(seq, PPT_START, PPT_END, rng), AG_START)
    both = m_acc(corrupt_region(seq, BP_START, PPT_END, rng), AG_START)
    rand_ag = m_acc(random_ag_variant(seq, AG_START, AG_LEN, rng), AG_START)
    decoy_pos = find_decoy_ag(seq, DECOY_FROM, DECOY_TO)
    decoy = m_acc(seq, decoy_pos) if decoy_pos is not None else None
    return dict(clean=clean, bp_only=bp_only, ppt_only=ppt_only, both=both,
                rand_ag=rand_ag, decoy=decoy, has_decoy=decoy_pos is not None)


def score_negative_sample(seq: str) -> dict:
    rng = pyrandom.Random(hash(seq) & 0xFFFFFFFF)
    clean = m_acc(seq, AG_START)
    bp_only = m_acc(corrupt_region(seq, BP_START, BP_END, rng), AG_START)
    ppt_only = m_acc(corrupt_region(seq, PPT_START, PPT_END, rng), AG_START)
    both = m_acc(corrupt_region(seq, BP_START, PPT_END, rng), AG_START)
    return dict(clean=clean, bp_only=bp_only, ppt_only=ppt_only, both=both)


acc_rows = [score_acceptor_sample(s) for s in tqdm(acceptor_df["sequence"], desc="acceptor windows")]
neg_rows = [score_negative_sample(s) for s in tqdm(negative_df["sequence"], desc="non-splice controls")]

acc_pd = pd.DataFrame(acc_rows)
neg_pd = pd.DataFrame(neg_rows)
acc_pd.to_csv("phase0_per_sample_acceptor.csv", index=False)
neg_pd.to_csv("phase0_per_sample_negative.csv", index=False)


In [ ]:
# --- bootstrap CIs + summary table ---------------------------------------
def bootstrap_ci(values, n_boot=N_BOOT, alpha=ALPHA, seed=SEED):
    values = np.asarray(values, dtype=float)
    if len(values) == 0:
        return float("nan"), float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(values), size=(n_boot, len(values)))
    boot_means = values[idx].mean(axis=1)
    lo, hi = np.percentile(boot_means, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(values.mean()), float(lo), float(hi)


def summarize(label, diffs):
    mean, lo, hi = bootstrap_ci(diffs)
    print(f"  {label:36s} {mean:+.3f}  [{lo:+.3f}, {hi:+.3f}]  (n={len(diffs)})")
    return {"mean": mean, "ci_lo": lo, "ci_hi": hi, "n": len(diffs)}


print("\nresults (mean Delta, 95% bootstrap CI):")
summary = {}
summary["delta_ppt_bp_only"] = summarize("Delta_ppt (BP disruption)", acc_pd["clean"] - acc_pd["bp_only"])
summary["delta_ppt_control_nonsplice"] = summarize("  -> non-splice control", neg_pd["clean"] - neg_pd["bp_only"])
summary["context_required"] = summarize("context required (clean - random-AG)", acc_pd["clean"] - acc_pd["rand_ag"])

decoy_df = acc_pd[acc_pd["has_decoy"]]
summary["ag_scanning"] = summarize("AG-scanning (true - decoy)", decoy_df["clean"] - decoy_df["decoy"])

summary["ppt_only_alone"] = summarize("PPT-only disruption alone", acc_pd["clean"] - acc_pd["ppt_only"])
summary["ppt_only_control_nonsplice"] = summarize("  -> non-splice control", neg_pd["clean"] - neg_pd["ppt_only"])
summary["both_disrupted"] = summarize("BP+PPT both disrupted", acc_pd["clean"] - acc_pd["both"])
summary["both_control_nonsplice"] = summarize("  -> non-splice control", neg_pd["clean"] - neg_pd["both"])

import json
with open("phase0_results.json", "w") as f:
    json.dump(summary, f, indent=2)
print("\nwrote phase0_results.json, phase0_per_sample_{acceptor,negative}.csv")


In [ ]:
# optional: pull the result files down from Colab
# from google.colab import files
# files.download("phase0_results.json")
# files.download("phase0_per_sample_acceptor.csv")
# files.download("phase0_per_sample_negative.csv")
